<a href="https://colab.research.google.com/github/Shun0212/CodeBERTPretrained/blob/main/UseMyCodeMorph_ModernBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers>=4.48.0
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.0 MB/s eta 0:00:00


In [2]:
!pip install transformers huggingface_hub
from huggingface_hub import notebook_login

# Hugging Face にログイン
notebook_login()

In [3]:
from transformers import AutoModel, AutoTokenizer
import os
import sys
import torch
import random
from datasets import load_dataset, Dataset
from tokenizers import BertWordPieceTokenizer
from transformers import ModernBertConfig, ModernBertForMaskedLM, PreTrainedTokenizerFast, DataCollatorForLanguageModeling, Trainer, TrainingArguments


# モデル名
repo_name = "Shuu12121/CodeMorph-ModernBERT"
# Hugging Face からモデルをロード
model = ModernBertForMaskedLM.from_pretrained(repo_name)
tokenizer = AutoTokenizer.from_pretrained(repo_name)

print("モデルのロード成功！")
print(model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/408k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
     

ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
      (1-11): 11

In [4]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)
print(fill_mask("def add_numbers(a, b): return a + [MASK]"))

Device set to use cuda:0


[{'score': 0.9981516003608704, 'token': 49908, 'token_str': 'b', 'sequence': 'def add _ numbers ( a, b ) : return a + b'}, {'score': 0.0010269464692100883, 'token': 49886, 'token_str': 'a', 'sequence': 'def add _ numbers ( a, b ) : return a + a'}, {'score': 0.00034104351652786136, 'token': 49891, 'token_str': 'c', 'sequence': 'def add _ numbers ( a, b ) : return a + c'}, {'score': 8.821619121590629e-05, 'token': 49938, 'token_str': '1', 'sequence': 'def add _ numbers ( a, b ) : return a + 1'}, {'score': 2.8221440516063012e-05, 'token': 159, 'token_str': 'end', 'sequence': 'def add _ numbers ( a, b ) : return a + end'}]


In [10]:
import torch

def get_embedding(text, model, tokenizer, device="cuda"):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    # token_type_ids があれば削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding

embedding = get_embedding("def my_function(): pass", model, tokenizer)
print(embedding.shape)


torch.Size([1, 768])


In [12]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデルでコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットのロード
    # ------------------------------
    print("\ngoogle/code_x_glue_tc_nl_code_search_adv データセット (Test) をロードします...")
    dataset = load_dataset("google/code_x_glue_tc_nl_code_search_adv", split="test", trust_remote_code=True)
    subset_tc = dataset.select(range(19210))
    max_examples = 100  # 実験用に先頭100サンプルを利用
    subset = dataset.select(range(max_examples))

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/codebert-base-mlm", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-py", "class": T5ForConditionalGeneration},
        {"name": "Salesforce/codet5-large-ntp-py", "class": T5ForConditionalGeneration},
        {"name": "Shuu12121/CodeMorph-BERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-BERTv2", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
    ]

    for config in model_configs:
        model_name = config["name"]
        model_class = config["class"]
        print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name)
        model.to(device)

        metrics = evaluate_code_search(model, tokenizer, subset, device,
                                       max_examples=max_examples,
                                       pool_size=100,
                                       query_field="docstring",
                                       code_field="code")
        display_code_search_results(metrics, model_name)


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/728k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/254M [00:00<?, ?B/s]

データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-BERTv2 Code Search Evaluation Results ====
MRR:         0.2790
MAP:         0.2790
R-Precision: 0.1900

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.1900    0.1900    0.1900    0.1900    0.1900         0.1900         
5     0.3700    0.2480    0.2779    0.2760    0.3700         0.3700         
10    0.4700    0.2608    0.3098    0.2987    0.4700         0.4700         
50    0.7900    0.2761    0.3805    0.3276    0.7900         0.7900         
100   1.0000    0.2790    0.4142    0.3332    1.0000         1.0000         

Shuu12121/CodeMorph-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT Code Search Evaluation Results ====
MRR:         0.5236
MAP:         0.5236
R-Precision: 0.3800

K